Multi-agent Customer Support Automation

In [1]:
!pip install crewai crewai_tools langchain_community

INFO: pip is looking at multiple versions of kubernetes to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 MB 35.9 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 89.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.6/803.6 kB 38.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 54.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.4/867.4 kB 38.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [3]:
from crewai import Agent, Task, Crew

In [4]:
import os
from getpass import getpass

# Enter OpenAI API key securely
api_key = getpass("Enter your OpenAI API key: ")

# Set environment variables
os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o"

Enter your OpenAI API key: ··········


In [6]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	llm="gpt-4o",
	allow_delegation=False,
	verbose=True
)

In [7]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	), #By not setting allow_delegation=False, allow_delegation takes its default value of being True.
	llm="gpt-4o",
	verbose=True
)


    * Role Playing: Both agents have been given a role, goal and backstory.
    * Focus: Both agents have been prompted to get into the character of the roles they are playing.
    * Cooperation: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.



In [8]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool,WebsiteSearchTool

In [9]:
#CrewAI user manual
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)

Creating Tasks

In [10]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

In [11]:
#Here the QA Agent will only review the work of the Support Agent
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


In [13]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True,
  memory=True
)

In [15]:
inputs = {
    "customer": "Learn AI with Anand",
    "person": "Dr.Anand",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}
result = await crew.kickoff_async(inputs=inputs)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7e11b677-13db-4fd4-98ae-df8a11bd5b93                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Learn AI with Anand just reached out with a super important ask:                                         │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Dr.Anand from Learn AI with Anand is the one that reached out. Make sure to use everything you know to         │
│  provide the best support possible.You must strive to provide a complete and accurate response to the           │
│  customer's inquiry.                                                                                            │
│  ID: 6050490e-01ab-45aa-b177-e7741d7f1de1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Task: Learn AI with Anand just reached out with a super important ask:                                         │
│  I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you   │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Dr.Anand from Learn AI with Anand is the one that reached out. Make sure to use everything you know to         │
│  provide the best support possible.You must strive to provide a complete and accurate response to the           │
│  customer's inquiry.                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_website_content                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: read_website_content                                                                                     │
│  Output: The following text is scraped website content:                                                         │
│  CrewAI Documentation - CrewAI                                                                                  │
│  Documentation Index Fetch the complete documentation index at: /llms.txt Use this file to discover all         │
│  available pages before exploring further. Skip to main content Assistant Responses are generated using AI and  │
│  may contain mistakes. CrewAI home page v1.15.33 English Search... ⌘ K Ask Assistant ⌘ I Start Cloud Trial      │
│  crewAIInc/crewAI crewAIInc/crewAI Search... Navigation Welcome CrewAI Documentation Home Documentation AMP     │
│  API Reference Examples Changelog Website Forum Blog CrewGPT Welcome CrewAI Documentation Welcome CrewAI        │
│  Documentation Copy page Copy page Build collaborative AI agents, crews, and flows — production ready from day  │
│  one. Copy page Copy page ​ Ship multi‑agent systems with confidence Design agents, orchestrate crews, and       │
│  automate flows with guardrails, memory, knowledge, and observability baked in. Get started Copy agent setup    │
│  prompt Coding-agent guide API Reference                                                                        │
│  ​ Get started                                                                                                   │
│  Introduction Overview of CrewAI concepts, architecture, and what you can build with agents, crews, and flows.  │
│  Installation Install via uv , configure API keys, and set up the CLI for local development. Quickstart Spin    │
│  up your first crew in minutes. Learn the core runtime, project layout, and dev loop.                           │
│  ​ Build the basics                                                                                              │
│  Agents Compose agents with tools, memory, knowledge, and structured outputs using Pydantic. Includes           │
│  templates and best practices. Flows Orchestrate start/listen/router steps, manage state, persist execution,    │
│  and resume long-running workflows. Tasks & Processes Define sequential, hierarchical, or hybrid processes      │
│  with guardrails, callbacks, and human-in-the-loop triggers.                                                    │
│  ​ Enterprise journey                                                                                            │
│  Deploy automations Manage environments, redeploy safely, and monitor live runs directly from the Enterprise    │
│  console. Triggers & Flows Connect Gmail, Slack, Salesforce, and more. Pass trigger payloads into crews and     │
│  flows automatically. Team management Invite teammates, configure RBAC, and control access to production        │
│  automations.                                                                                                   │
│  ​ What’s new                                                                                                    │
│  Triggers overview Unified overview for Gmail, Drive, Outlook, Teams, OneDrive, HubSpot, and more — now with    │
│  sample payloads and crews. Integration tools Call existing CrewAI automations or Amazon Bedrock Agents         │
│  directly from your crews using the updated integration toolkit.                                                │
│  Browse the examples and cookbooks for end-to-end reference implementations across agents, flows, and           │
│  enterprise automations.                          

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello Dr. Anand!                                                                                               │
│                                                                                                                 │
│  I'm thrilled to assist you with setting up a Crew and adding memory to it. Let’s walk through the process      │
│  together to ensure you have everything you need to begin building with CrewAI:                                 │
│                                                                                                                 │
│  ### Setting Up a Crew in CrewAI                                                                                │
│                                                                                                                 │
│  1. **Introduction to CrewAI:**                                                                                 │
│     CrewAI is designed to help you create collaborative AI agents, crews, and workflows that are ready for      │
│  real-world applications. It's a dynamic platform that allows you to design, orchestrate, and automate with     │
│  built-in functionalities such as memory, knowledge, and observability.                                         │
│                                                                                                                 │
│  2. **Installation and Quickstart:**                                                                            │
│     - **Install CrewAI**: We recommend using UV for installation, which is simple and straightforward. Once     │
│  installed, configure your API keys and set up the Command Line Interface (CLI) for development on your local   │
│  environment.                                                                                                   │
│     - **Quickstart Guide**: Understand the core runtime, get familiar with the project layout, and start your   │
│  development loop to have your first crew running in just a few minutes.                                        │
│                                                                                                                 │
│  ### Adding Memory to Your Crew                                                                                 │
│                                                                                                                 │
│  3. **Agents Composition:**                                                                                     │
│     When creating agents in your crew, integrating tools like memory, knowledge, and structured outputs is      │
│  essential. This is usually achieved with Pydantic, a Python library for data validation and management.        │
│                                                                                                                 │
│  4. **Using Memory:**                                                                                           │
│     - Memory is a critical component of your agents and crew configuration. It allows your agents to retain     │
│  past interactions, contextual details, and relevant data, thus enhancing decision-making and improving         │
│  interaction quality.                                                                                           │
│     - To add memory, define it explicitly in your agent

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the response drafted by the Senior Support Representative for Learn AI with Anand's inquiry.      │
│  Ensure that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for      │
│  customer support.                                                                                              │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7e11b677-13db-4fd4-98ae-df8a11bd5b93                                                                       │
│  Final Output: Hello Dr. Anand!                                                                                 │
│                                                                                                                 │
│  I'm thrilled to assist you with setting up a Crew and adding memory to it. Let’s walk through the process      │
│  together to ensure you have everything you need to begin building with CrewAI:                                 │
│                                                                                                                 │
│  ### Setting Up a Crew in CrewAI                                                                                │
│                                                                                                                 │
│  1. **Introduction to CrewAI:**                                                                                 │
│     CrewAI is designed to help you create collaborative AI agents, crews, and workflows that are ready for      │
│  real-world applications. It's a dynamic platform that allows you to design, orchestrate, and automate with     │
│  built-in functionalities such as memory, knowledge, and observability.                                         │
│                                                                                                                 │
│  2. **Installation and Quickstart:**                                                                            │
│     - **Install CrewAI**: We recommend using UV for installation, which is simple and straightforward. Once     │
│  installed, configure your API keys and set up the Command Line Interface (CLI) for development on your local   │
│  environment.                                                                                                   │
│     - **Quickstart Guide**: Understand the core runtime, get familiar with the project layout, and start your   │
│  development loop to have your first crew running in just a few minutes.                                        │
│                                                                                                                 │
│  ### Adding Memory to Your Crew                                                                                 │
│                                                                                                                 │
│  3. **Agents Composition:**                                                                                     │
│     When creating agents in your crew, integrating tools like memory, knowledge, and structured outputs is      │
│  essential. This is usually achieved with Pydantic, a Python library for data validation and management.        │
│                                                                                                                 │
│  4. **Using Memory:**                                                                                           │
│     - Memory is a critical component of your agents and crew configuration. It allows your agents to retain     │
│  past interactions, contextual details, and relevant data, thus enhancing decision-making and improving         │
│  interaction quality.                                                                                           │
│     - To add memory, define it explicitly in your agen

Hello Dr. Anand! 

I'm thrilled to assist you with setting up a Crew and adding memory to it. Let’s walk through the process together to ensure you have everything you need to begin building with CrewAI:

### Setting Up a Crew in CrewAI

1. **Introduction to CrewAI:**
   CrewAI is designed to help you create collaborative AI agents, crews, and workflows that are ready for real-world applications. It's a dynamic platform that allows you to design, orchestrate, and automate with built-in functionalities such as memory, knowledge, and observability.

2. **Installation and Quickstart:**
   - **Install CrewAI**: We recommend using UV for installation, which is simple and straightforward. Once installed, configure your API keys and set up the Command Line Interface (CLI) for development on your local environment.
   - **Quickstart Guide**: Understand the core runtime, get familiar with the project layout, and start your development loop to have your first crew running in just a few minutes.
